# LeetCode 770 — Basic Calculator IV bằng Python

- Link: https://leetcode.com/problems/basic-calculator-iv/
- Độ khó: **Hard**
- Chủ đề: Phân tích cú pháp (parsing), Đệ quy đi xuống (recursive descent), Ngăn xếp (shunting-yard), Hash Map biểu diễn đa thức, Nạp chồng toán tử

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def basicCalculatorIV(self, expression: str, evalvars: List[str], evalints: List[int]) -> List[str]:
```

## 1. Giới thiệu & mục tiêu học

Các bài Basic Calculator I–III tính ra **một con số**. Bài IV khác hẳn: biểu thức chứa **biến**, và ta phải **rút gọn thành đa thức** (khai triển, gom các hạng tử đồng dạng), rồi in theo một thứ tự quy định. Đây là bài tổng hợp hai kỹ năng:

1. **Phân tích cú pháp** biểu thức có độ ưu tiên `*` > `+ -` và dấu ngoặc.
2. **Biểu diễn đa thức** bằng `dict` và cài các phép cộng, trừ, nhân.

Sau notebook này bạn sẽ:

1. Biểu diễn đa thức nhiều biến bằng `dict`: khoá là **bộ biến đã sắp** (đơn thức), giá trị là **hệ số**.
2. Viết bộ phân tích **đệ quy đi xuống** (expr → term → factor).
3. Viết phiên bản **hai ngăn xếp** (shunting-yard) — không đệ quy.
4. Dùng **nạp chồng toán tử** (`__add__`, `__mul__`) để "mượn" bộ phân tích của Python.
5. Định dạng kết quả đúng quy tắc: bậc giảm dần, cùng bậc thì theo thứ tự từ điển của danh sách biến; bỏ hạng tử hệ số 0.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho chuỗi `expression` gồm: số nguyên không âm, tên biến (chữ thường), các toán tử `+`, `-`, `*`, dấu ngoặc `(` `)`, các token cách nhau bởi **một dấu cách**. Cho thêm `evalvars` và `evalints`: biến `evalvars[i]` được **thay** bằng giá trị `evalints[i]`. Hãy rút gọn biểu thức và trả về danh sách các hạng tử dạng chuỗi:

- Mỗi hạng tử: `hệ số` rồi tới các biến, nối bằng `*`, biến trong một hạng tử **sắp theo từ điển**. Ví dụ `"-2*a*b*c"`, hằng số `"5"`.
- Thứ tự hạng tử: **bậc** (số biến, tính cả lặp) **giảm dần**; cùng bậc thì so sánh **danh sách biến** theo thứ tự từ điển. Hằng số đứng cuối.
- Bỏ các hạng tử có hệ số 0. Hệ số luôn được in, kể cả `1` và `-1` (ví dụ `"1*e*e"`).

**Ví dụ 1**

```text
Input: expression = "e + 8 - a + 5", evalvars = ["e"], evalints = [1]
Output: ["-1*a","14"]
```

**Ví dụ 2**

```text
Input: expression = "e - 8 + temperature - pressure", evalvars = ["e", "temperature"], evalints = [1, 12]
Output: ["-1*pressure","5"]
```

**Ví dụ 3**

```text
Input: expression = "(e + 8) * (e - 8)", evalvars = [], evalints = []
Output: ["1*e*e","-64"]
```

**Ví dụ thêm** (thường gặp trong test): `"7 - 7"` → `[]`; `"a * b * c + b * a * c * 4"` → `["5*a*b*c"]`.

**Ràng buộc**

- `1 <= expression.length <= 250`
- `expression` gồm chữ thường, chữ số, `'+'`, `'-'`, `'*'`, `'('`, `')'`, `' '`; các token cách nhau đúng một dấu cách; không có dấu trừ một ngôi.
- `0 <= evalvars.length <= 100`, `1 <= evalvars[i].length <= 20`, `evalints.length == evalvars.length`, `-100 <= evalints[i] <= 100`.

In [1]:
from typing import List

EXAMPLES = [
    ("e + 8 - a + 5", ["e"], [1], ["-1*a", "14"]),
    ("e - 8 + temperature - pressure", ["e", "temperature"], [1, 12], ["-1*pressure", "5"]),
    ("(e + 8) * (e - 8)", [], [], ["1*e*e", "-64"]),
    ("7 - 7", [], [], []),
    ("a * b * c + b * a * c * 4", [], [], ["5*a*b*c"]),
    ("((a - b) * (b - c) + (c - a)) * ((a - b) + (b - c) * (c - a))", [], [],
     ["-1*a*a*b*b", "2*a*a*b*c", "-1*a*a*c*c", "1*a*b*b*b", "-1*a*b*b*c", "-1*a*b*c*c",
      "1*a*c*c*c", "-1*b*b*b*c", "2*b*b*c*c", "-1*b*c*c*c", "2*a*a*b", "-2*a*a*c", "-2*a*b*b",
      "2*a*c*c", "1*b*b*b", "-1*b*b*c", "1*b*c*c", "-1*c*c*c", "-1*a*a", "1*a*b", "1*a*c", "-1*b*c"]),
]
print(len(EXAMPLES), "ví dụ")

6 ví dụ


## 3. Kiến thức nền

### 3.1. Tách token bằng biểu thức chính quy

`re.findall(r'\d+|[a-z]+|[()+\-*]', s)` lấy ra số, tên biến, toán tử và ngoặc — không phụ thuộc vào dấu cách.

In [2]:
import re
TOKEN_RE = re.compile(r'\d+|[a-z]+|[()+\-*]')
print(TOKEN_RE.findall("(e + 8) * (e - 8)"))
print(TOKEN_RE.findall("e - 8 + temperature - pressure"))

['(', 'e', '+', '8', ')', '*', '(', 'e', '-', '8', ')']
['e', '-', '8', '+', 'temperature', '-', 'pressure']


### 3.2. Đa thức nhiều biến bằng `dict`

- **Đơn thức** (monomial) = tích các biến, biểu diễn bằng **tuple biến đã sắp**: `a*b*a` → `('a', 'a', 'b')`. Hằng số → tuple rỗng `()`.
- **Đa thức** = `dict` {đơn thức: hệ số}. Ví dụ `3*a*b - 2*a + 7` → `{('a','b'): 3, ('a',): -2, (): 7}`.
- Tuple **bất biến** nên làm khoá dict được; sắp biến giúp `a*b` và `b*a` trùng khoá.

In [3]:
p = {('a', 'b'): 3, ('a',): -2, (): 7}
q = {('a',): 2, (): 1}
print(p, q)
print(tuple(sorted(('b', 'a') + ('a',))))     # nhân hai đơn thức

{('a', 'b'): 3, ('a',): -2, (): 7} {('a',): 2, (): 1}
('a', 'a', 'b')


### 3.3. Các phép toán trên đa thức

- **Cộng/trừ**: cộng/trừ hệ số theo từng khoá; bỏ khoá có hệ số 0.
- **Nhân**: với mọi cặp đơn thức `(m1, c1)`, `(m2, c2)`: khoá `sorted(m1 + m2)`, cộng thêm `c1 * c2`.

In [4]:
def p_add(p, q, sign=1):
    r = dict(p)
    for m, c in q.items():
        r[m] = r.get(m, 0) + sign * c
        if r[m] == 0:
            del r[m]
    return r

def p_mul(p, q):
    r = {}
    for m1, c1 in p.items():
        for m2, c2 in q.items():
            m = tuple(sorted(m1 + m2))
            r[m] = r.get(m, 0) + c1 * c2
    return {m: c for m, c in r.items() if c != 0}

print("p + q =", p_add(p, q))
print("p - q =", p_add(p, q, -1))
print("(e+8)(e-8) =", p_mul({('e',): 1, (): 8}, {('e',): 1, (): -8}))

p + q = {('a', 'b'): 3, (): 8}
p - q = {('a', 'b'): 3, ('a',): -4, (): 6}
(e+8)(e-8) = {('e', 'e'): 1, (): -64}


### 3.4. Sắp xếp và định dạng kết quả

Khoá sắp: `(-len(m), m)` — bậc lớn trước; cùng bậc thì so sánh tuple biến theo từ điển (Python so sánh tuple chuỗi đúng như yêu cầu). Hằng số có `len = 0` nên tự động đứng cuối.

In [5]:
def p_format(p):
    out = []
    for m in sorted(p, key=lambda m: (-len(m), m)):
        if p[m] != 0:
            out.append('*'.join([str(p[m])] + list(m)))
    return out

print(p_format({(): 7, ('a',): -2, ('a', 'b'): 3, ('b', 'b'): 1, ('a', 'c'): 5}))

['3*a*b', '5*a*c', '1*b*b', '-2*a', '7']


### 3.5. Văn phạm biểu thức và độ ưu tiên

```text
expr   := term   (('+' | '-') term)*
term   := factor ('*' factor)*
factor := NUMBER | VARIABLE | '(' expr ')'
```

Mỗi quy tắc thành một hàm. `*` nằm "sâu hơn" `+ -` nên được thực hiện trước — đó là cách văn phạm mã hoá độ ưu tiên. Ví dụ nhỏ với số nguyên:

In [6]:
def eval_int(s):
    toks = TOKEN_RE.findall(s)
    i = 0
    def expr():
        nonlocal i
        v = term()
        while i < len(toks) and toks[i] in '+-':
            op = toks[i]; i += 1
            w = term()
            v = v + w if op == '+' else v - w
        return v
    def term():
        nonlocal i
        v = factor()
        while i < len(toks) and toks[i] == '*':
            i += 1
            v *= factor()
        return v
    def factor():
        nonlocal i
        t = toks[i]; i += 1
        if t == '(':
            v = expr(); i += 1      # bỏ ')'
            return v
        return int(t)
    return expr()

print(eval_int("2 + 3 * (4 - 1) * 2 - 5"), 2 + 3 * (4 - 1) * 2 - 5)

15 15


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Tính tay ví dụ 3: `(e + 8) * (e - 8)`

- `(e + 8)` → `{('e',): 1, (): 8}`.
- `(e - 8)` → `{('e',): 1, (): -8}`.
- Nhân: `e·e = ('e','e')`: 1; `e·(-8)` và `8·e` → `('e',)`: `-8 + 8 = 0` → **bỏ**; `8·(-8) = -64`.
- Kết quả `{('e','e'): 1, (): -64}` → `["1*e*e", "-64"]`.

### 4.2. Tính tay ví dụ 1 với thay biến: `e + 8 - a + 5`, `e = 1`

Khi gặp token `e` (có trong `evalvars`) ta coi nó là **hằng số 1** ngay lúc đọc. Biểu thức thành `1 + 8 - a + 5` → `{(): 14, ('a',): -1}` → `["-1*a", "14"]`.

### 4.3. Thứ tự in

Ví dụ `{('b','b'):1, ('a','c'):5, ('a','b'):3, ('a',):-2, ():7}`:

- Bậc 2: `('a','b')`, `('a','c')`, `('b','b')` — so sánh tuple: `a,b < a,c < b,b`.
- Bậc 1: `('a',)`. Bậc 0: hằng số.

→ `["3*a*b", "5*a*c", "1*b*b", "-2*a", "7"]`.

In [7]:
print(p_format({('b', 'b'): 1, ('a', 'c'): 5, ('a', 'b'): 3, ('a',): -2, (): 7}))

['3*a*b', '5*a*c', '1*b*b', '-2*a', '7']


## 5. Cách 1 — Brute Force: khai triển thành danh sách hạng tử **chưa gom**, chỉ gom ở cuối

**Ý tưởng.** Biểu diễn mỗi biểu thức con bằng **danh sách** các cặp `(hệ số, đơn thức)` — không gom hạng tử đồng dạng. Cộng = nối danh sách; trừ = nối danh sách đã đổi dấu; nhân = tích Descartes. Chỉ ở bước cuối mới gom vào `dict`. Đúng nhưng danh sách có thể **phình mũ** khi nhân nhiều nhân tử.

In [8]:
def brute(expression, evalvars, evalints):
    env = dict(zip(evalvars, evalints))
    toks = TOKEN_RE.findall(expression)
    i = 0

    def atom(t):
        if t.isdigit():
            return [(int(t), ())]
        if t in env:
            return [(env[t], ())]
        return [(1, (t,))]

    def expr():
        nonlocal i
        v = term()
        while i < len(toks) and toks[i] in '+-':
            op = toks[i]; i += 1
            w = term()
            v = v + (w if op == '+' else [(-c, m) for c, m in w])
        return v

    def term():
        nonlocal i
        v = factor()
        while i < len(toks) and toks[i] == '*':
            i += 1
            w = factor()
            v = [(c1 * c2, m1 + m2) for c1, m1 in v for c2, m2 in w]   # chưa gom!
        return v

    def factor():
        nonlocal i
        t = toks[i]; i += 1
        if t == '(':
            v = expr(); i += 1
            return v
        return atom(t)

    terms = expr()
    poly = {}
    for c, m in terms:
        key = tuple(sorted(m))
        poly[key] = poly.get(key, 0) + c
    return p_format(poly)

for expr_, ev, ei, exp in EXAMPLES:
    print(brute(expr_, ev, ei) == exp, brute(expr_, ev, ei)[:4])

True ['-1*a', '14']
True ['-1*pressure', '5']
True ['1*e*e', '-64']
True []
True ['5*a*b*c']
True ['-1*a*a*b*b', '2*a*a*b*c', '-1*a*a*c*c', '1*a*b*b*b']


**Độ phức tạp.** Nếu biểu thức là tích của `k` nhân tử, mỗi nhân tử có `t` hạng tử, danh sách chưa gom có `t^k` phần tử — **mũ**. Ví dụ `(a + b) * (a + b) * ... ` (10 lần) cho `2^10 = 1024` phần tử, trong khi đa thức gom lại chỉ có 11 hạng tử.

**Nút thắt.** Không gom sớm → lặp lại công việc trên các hạng tử đồng dạng. Cải tiến: **gom ngay sau mỗi phép toán** bằng `dict`.

In [9]:
e10 = " * ".join(["(a + b)"] * 10)
import time
t0 = time.perf_counter(); r = brute(e10, [], []); t1 = time.perf_counter()
print(len(r), "hạng tử sau gom;", f"{1000 * (t1 - t0):.1f} ms")

11 hạng tử sau gom; 1.7 ms


## 6. Bước đột phá

**Insight 1 — Đa thức là "giá trị".** Hãy coi đa thức (dict) như một **con số mới**: bộ phân tích biểu thức của Basic Calculator III vẫn dùng nguyên vẹn, chỉ thay phép `+ - *` trên số bằng `p_add`, `p_mul` trên đa thức. Mỗi kết quả trung gian luôn được **gom** → kích thước bị chặn bởi số đơn thức phân biệt.

**Insight 2 — Thay biến ngay khi đọc token.** Biến có trong `evalvars` trở thành hằng số `{(): giá trị}` (bỏ nếu giá trị 0 — `{}`), biến khác thành `{(name,): 1}`, số thành `{(): n}`.

**Insight 3 — Đơn thức chuẩn hoá.** Luôn lưu đơn thức là **tuple đã sắp**, để `a*b` và `b*a` gộp được và để việc sắp kết quả cuối cùng dùng luôn so sánh tuple.

**Quy tắc nếu… thì…**

- Nếu token là số → `{(): int(t)}` (bỏ nếu 0).
- Nếu token là biến trong `evalvars` → hằng số tương ứng.
- Nếu token là biến khác → `{(t,): 1}`.
- Nếu gặp `*` → nhân ngay với nhân tử kế tiếp (ưu tiên cao).
- Nếu gặp `+`/`-` → cộng/trừ với **cả hạng** (term) kế tiếp.
- Hệ số về 0 → **xoá** khỏi dict.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy đi xuống trên đa thức (dict)

**Giải thích.** Đúng văn phạm ở mục 3.5, giá trị là dict.

In [10]:
def make_atom(t, env):
    if t.isdigit():
        v = int(t)
        return {(): v} if v else {}
    if t in env:
        v = env[t]
        return {(): v} if v else {}
    return {(t,): 1}

def calc_rd(expression, evalvars, evalints):
    env = dict(zip(evalvars, evalints))
    toks = TOKEN_RE.findall(expression)
    pos = 0

    def expr():
        nonlocal pos
        v = term()
        while pos < len(toks) and toks[pos] in '+-':
            op = toks[pos]; pos += 1
            v = p_add(v, term(), 1 if op == '+' else -1)
        return v

    def term():
        nonlocal pos
        v = factor()
        while pos < len(toks) and toks[pos] == '*':
            pos += 1
            v = p_mul(v, factor())
        return v

    def factor():
        nonlocal pos
        t = toks[pos]; pos += 1
        if t == '(':
            v = expr()
            pos += 1          # bỏ qua ')'
            return v
        return make_atom(t, env)

    return p_format(expr())

for expr_, ev, ei, exp in EXAMPLES:
    print(calc_rd(expr_, ev, ei) == exp)

True
True
True
True
True
True


**Code nộp LeetCode (Cách 2).**

In [11]:
class SolutionRD:
    def basicCalculatorIV(self, expression: str, evalvars: List[str], evalints: List[int]) -> List[str]:
        import re
        env = dict(zip(evalvars, evalints))
        toks = re.findall(r'\d+|[a-z]+|[()+\-*]', expression)
        self.pos = 0

        def add(p, q, sign):
            r = dict(p)
            for m, c in q.items():
                r[m] = r.get(m, 0) + sign * c
                if r[m] == 0:
                    del r[m]
            return r

        def mul(p, q):
            r = {}
            for m1, c1 in p.items():
                for m2, c2 in q.items():
                    m = tuple(sorted(m1 + m2))
                    r[m] = r.get(m, 0) + c1 * c2
            return {m: c for m, c in r.items() if c}

        def atom(t):
            if t.isdigit():
                v = int(t)
            elif t in env:
                v = env[t]
            else:
                return {(t,): 1}
            return {(): v} if v else {}

        def expr():
            v = term()
            while self.pos < len(toks) and toks[self.pos] in '+-':
                op = toks[self.pos]; self.pos += 1
                v = add(v, term(), 1 if op == '+' else -1)
            return v

        def term():
            v = factor()
            while self.pos < len(toks) and toks[self.pos] == '*':
                self.pos += 1
                v = mul(v, factor())
            return v

        def factor():
            t = toks[self.pos]; self.pos += 1
            if t == '(':
                v = expr()
                self.pos += 1
                return v
            return atom(t)

        poly = expr()
        return ['*'.join([str(poly[m])] + list(m))
                for m in sorted(poly, key=lambda m: (-len(m), m))]

print([SolutionRD().basicCalculatorIV(e, v, i) == exp for e, v, i, exp in EXAMPLES])

[True, True, True, True, True, True]


**Độ phức tạp.** Gọi `T` là số token, `M` là số đơn thức tối đa của một đa thức trung gian, `D` là bậc tối đa. Mỗi phép nhân `O(M^2 · D log D)` (sắp tuple). Tổng `O(T · M^2 · D log D)`. Với độ dài 250, `M` và `D` nhỏ trong thực tế.

### 7.2. Cách 3 — Hai ngăn xếp (shunting-yard), không đệ quy

**Giải thích.** Ngăn xếp `vals` (đa thức) và `ops` (toán tử, ngoặc mở). Độ ưu tiên: `*` = 2, `+ -` = 1.

- Token giá trị → đẩy vào `vals`.
- `(` → đẩy vào `ops`.
- `)` → thực hiện các toán tử cho tới khi gặp `(`; pop `(`.
- Toán tử `o` → trong khi đỉnh `ops` là toán tử có độ ưu tiên `>=` `o` thì thực hiện nó; rồi đẩy `o`.
- Hết token → thực hiện hết.

`>=` (chứ không `>`) đảm bảo **kết hợp trái**: `a - b - c = (a - b) - c`.

In [12]:
PREC = {'+': 1, '-': 1, '*': 2}

def apply_op(vals, op):
    b = vals.pop(); a = vals.pop()
    if op == '+':
        vals.append(p_add(a, b))
    elif op == '-':
        vals.append(p_add(a, b, -1))
    else:
        vals.append(p_mul(a, b))

def calc_stack(expression, evalvars, evalints):
    env = dict(zip(evalvars, evalints))
    vals, ops = [], []
    for t in TOKEN_RE.findall(expression):
        if t == '(':
            ops.append(t)
        elif t == ')':
            while ops[-1] != '(':
                apply_op(vals, ops.pop())
            ops.pop()
        elif t in PREC:
            while ops and ops[-1] != '(' and PREC[ops[-1]] >= PREC[t]:
                apply_op(vals, ops.pop())
            ops.append(t)
        else:
            vals.append(make_atom(t, env))
    while ops:
        apply_op(vals, ops.pop())
    return p_format(vals[-1])

for expr_, ev, ei, exp in EXAMPLES:
    print(calc_stack(expr_, ev, ei) == exp)

True
True
True
True
True
True


**Code nộp LeetCode (Cách 3).**

In [13]:
class SolutionStack:
    def basicCalculatorIV(self, expression: str, evalvars: List[str], evalints: List[int]) -> List[str]:
        import re
        env = dict(zip(evalvars, evalints))
        prec = {'+': 1, '-': 1, '*': 2}

        def combine(a, b, op):
            if op == '*':
                r = {}
                for m1, c1 in a.items():
                    for m2, c2 in b.items():
                        m = tuple(sorted(m1 + m2))
                        r[m] = r.get(m, 0) + c1 * c2
            else:
                s = 1 if op == '+' else -1
                r = dict(a)
                for m, c in b.items():
                    r[m] = r.get(m, 0) + s * c
            return {m: c for m, c in r.items() if c}

        vals, ops = [], []

        def reduce_once():
            b = vals.pop(); a = vals.pop()
            vals.append(combine(a, b, ops.pop()))

        for t in re.findall(r'\d+|[a-z]+|[()+\-*]', expression):
            if t == '(':
                ops.append(t)
            elif t == ')':
                while ops[-1] != '(':
                    reduce_once()
                ops.pop()
            elif t in prec:
                while ops and ops[-1] != '(' and prec[ops[-1]] >= prec[t]:
                    reduce_once()
                ops.append(t)
            else:
                if t.isdigit():
                    v = int(t)
                    vals.append({(): v} if v else {})
                elif t in env:
                    vals.append({(): env[t]} if env[t] else {})
                else:
                    vals.append({(t,): 1})
        while ops:
            reduce_once()
        poly = vals[-1]
        return ['*'.join([str(poly[m])] + list(m))
                for m in sorted(poly, key=lambda m: (-len(m), m))]

print([SolutionStack().basicCalculatorIV(e, v, i) == exp for e, v, i, exp in EXAMPLES])

[True, True, True, True, True, True]


**Độ phức tạp.** Như Cách 2 (mỗi toán tử được thực hiện đúng một lần). Không có đệ quy → không lo giới hạn độ sâu khi ngoặc lồng nhiều.

### 7.3. Cách 4 — Nạp chồng toán tử + `eval` của Python

**Giải thích.** Python đã có sẵn bộ phân tích biểu thức với đúng độ ưu tiên `* > + -` và ngoặc. Ta viết lớp `Poly` có `__add__`, `__sub__`, `__mul__`, rồi **dịch** từng token: biến/số → `Poly(...)` (gọi qua tên hàm `P`), toán tử/ngoặc giữ nguyên. Sau đó `eval` chuỗi đã dịch trong một môi trường **chỉ** chứa `P`.

> Cảnh báo: `eval` nguy hiểm với input không tin cậy. Ở đây ta chỉ ghép những token đã được biểu thức chính quy lọc (số, tên `[a-z]+` biến thành chuỗi trích dẫn, toán tử), và `eval` với `{"__builtins__": {}}`. Trong phỏng vấn nên trình bày Cách 2/3; cách này để thấy sức mạnh của nạp chồng toán tử.

In [14]:
class Poly:
    __slots__ = ("t",)

    def __init__(self, terms=None):
        self.t = {m: c for m, c in (terms or {}).items() if c}

    def __add__(self, o):
        return Poly(p_add(self.t, o.t))

    def __sub__(self, o):
        return Poly(p_add(self.t, o.t, -1))

    def __mul__(self, o):
        return Poly(p_mul(self.t, o.t))

def calc_eval(expression, evalvars, evalints):
    env = dict(zip(evalvars, evalints))

    def P(tok):
        return Poly(make_atom(tok, env))

    code = []
    for t in TOKEN_RE.findall(expression):
        if t.isdigit() or t.isalpha():
            code.append(f"P({t!r})")
        else:
            code.append(t)
    poly = eval(' '.join(code), {"__builtins__": {}}, {"P": P})
    return p_format(poly.t)

print(' '.join(f"P({t!r})" if t.isalnum() else t for t in TOKEN_RE.findall("(e + 8) * (e - 8)")))
for expr_, ev, ei, exp in EXAMPLES:
    print(calc_eval(expr_, ev, ei) == exp)

( P('e') + P('8') ) * ( P('e') - P('8') )
True
True
True
True
True
True


**Code nộp LeetCode (Cách 4).**

In [15]:
class SolutionEval:
    def basicCalculatorIV(self, expression: str, evalvars: List[str], evalints: List[int]) -> List[str]:
        import re
        env = dict(zip(evalvars, evalints))

        class Poly:
            def __init__(self, d):
                self.d = {m: c for m, c in d.items() if c}
            def __add__(self, o):
                r = dict(self.d)
                for m, c in o.d.items():
                    r[m] = r.get(m, 0) + c
                return Poly(r)
            def __sub__(self, o):
                r = dict(self.d)
                for m, c in o.d.items():
                    r[m] = r.get(m, 0) - c
                return Poly(r)
            def __mul__(self, o):
                r = {}
                for m1, c1 in self.d.items():
                    for m2, c2 in o.d.items():
                        m = tuple(sorted(m1 + m2))
                        r[m] = r.get(m, 0) + c1 * c2
                return Poly(r)

        def P(t):
            if t.isdigit():
                return Poly({(): int(t)})
            if t in env:
                return Poly({(): env[t]})
            return Poly({(t,): 1})

        toks = re.findall(r'\d+|[a-z]+|[()+\-*]', expression)
        code = ' '.join(f"P({t!r})" if t.isalnum() else t for t in toks)
        poly = eval(code, {"__builtins__": {}}, {"P": P}).d
        return ['*'.join([str(poly[m])] + list(m))
                for m in sorted(poly, key=lambda m: (-len(m), m))]

print([SolutionEval().basicCalculatorIV(e, v, i) == exp for e, v, i, exp in EXAMPLES])

[True, True, True, True, True, True]


**Độ phức tạp.** Như Cách 2 (chỉ đổi người làm phân tích cú pháp).

## 8. TRACE từng bước

Trace **Cách 3 (hai ngăn xếp)** trên `"(e + 8) * (e - 8)"`. Ký hiệu đa thức gọn: `e+8` nghĩa là `{('e',):1, ():8}`.

| token | hành động | vals | ops |
|---|---|---|---|
| `(` | push ops | | `(` |
| `e` | push val | `e` | `(` |
| `+` | push ops | `e` | `( +` |
| `8` | push val | `e, 8` | `( +` |
| `)` | thực hiện `+` → `e+8`; pop `(` | `e+8` | |
| `*` | push ops | `e+8` | `*` |
| `(` | push ops | `e+8` | `* (` |
| `e`, `-`, `8` | ... | `e+8, e, 8` | `* ( -` |
| `)` | thực hiện `-` → `e-8`; pop `(` | `e+8, e-8` | `*` |
| hết | thực hiện `*` | `e*e - 64` | |

In [16]:
def show(p):
    return ' + '.join(f"{c}{'*' if m else ''}{'*'.join(m)}" for m, c in sorted(p.items(), key=lambda kv: (-len(kv[0]), kv[0]))) or "0"

def trace_stack(expression, evalvars=(), evalints=()):
    env = dict(zip(evalvars, evalints))
    vals, ops = [], []
    def red():
        op = ops.pop(); b = vals.pop(); a = vals.pop()
        r = p_mul(a, b) if op == '*' else p_add(a, b, 1 if op == '+' else -1)
        vals.append(r)
        return f"[{show(a)}] {op} [{show(b)}] = [{show(r)}]"
    for t in TOKEN_RE.findall(expression):
        acts = []
        if t == '(':
            ops.append(t)
        elif t == ')':
            while ops[-1] != '(':
                acts.append(red())
            ops.pop()
        elif t in PREC:
            while ops and ops[-1] != '(' and PREC[ops[-1]] >= PREC[t]:
                acts.append(red())
            ops.append(t)
        else:
            vals.append(make_atom(t, env))
        print(f"token {t:>3} | {'; '.join(acts) if acts else '':<40} | vals={[show(v) for v in vals]} ops={ops}")
    while ops:
        print("cuối:", red())
    print("kết quả:", p_format(vals[-1]))

trace_stack("(e + 8) * (e - 8)")

token   ( |                                          | vals=[] ops=['(']
token   e |                                          | vals=['1*e'] ops=['(']
token   + |                                          | vals=['1*e'] ops=['(', '+']
token   8 |                                          | vals=['1*e', '8'] ops=['(', '+']
token   ) | [1*e] + [8] = [1*e + 8]                  | vals=['1*e + 8'] ops=[]
token   * |                                          | vals=['1*e + 8'] ops=['*']
token   ( |                                          | vals=['1*e + 8'] ops=['*', '(']
token   e |                                          | vals=['1*e + 8', '1*e'] ops=['*', '(']
token   - |                                          | vals=['1*e + 8', '1*e'] ops=['*', '(', '-']
token   8 |                                          | vals=['1*e + 8', '1*e', '8'] ops=['*', '(', '-']
token   ) | [1*e] - [8] = [1*e + -8]                 | vals=['1*e + 8', '1*e + -8'] ops=['*']
cuối: [1*e + 8] * [1*e + -8] = [1*e*e +

Trace **Cách 2 (đệ quy)** — in lời gọi `expr/term/factor` với độ thụt theo độ sâu, trên `"a * b + 2 * (a - b) * a"`.

In [17]:
def trace_rd(expression):
    toks = TOKEN_RE.findall(expression)
    pos = 0
    depth = 0
    def log(msg):
        print("  " * depth + msg)
    def expr():
        nonlocal pos, depth
        log("expr"); depth += 1
        v = term()
        while pos < len(toks) and toks[pos] in '+-':
            op = toks[pos]; pos += 1
            w = term()
            v = p_add(v, w, 1 if op == '+' else -1)
            log(f"{op} -> [{show(v)}]")
        depth -= 1
        return v
    def term():
        nonlocal pos, depth
        log("term"); depth += 1
        v = factor()
        while pos < len(toks) and toks[pos] == '*':
            pos += 1
            w = factor()
            v = p_mul(v, w)
            log(f"* -> [{show(v)}]")
        depth -= 1
        return v
    def factor():
        nonlocal pos
        t = toks[pos]; pos += 1
        if t == '(':
            v = expr(); pos += 1
            return v
        log(f"factor {t}")
        return make_atom(t, {})
    res = expr()
    print("=>", p_format(res))

trace_rd("a * b + 2 * (a - b) * a")

expr
  term
    factor a
    factor b
    * -> [1*a*b]
  term
    factor 2
    expr
      term
        factor a
      term
        factor b
      - -> [1*a + -1*b]
    * -> [2*a + -2*b]
    factor a
    * -> [2*a*a + -2*a*b]
  + -> [2*a*a + -1*a*b]
=> ['2*a*a', '-1*a*b']


## 9. Code nộp LeetCode

Bản chọn: **Cách 2 (đệ quy đi xuống)** — rõ ràng, phản ánh trực tiếp văn phạm. (Nếu lo về độ sâu đệ quy với ngoặc lồng, dùng Cách 3; với độ dài ≤ 250 thì độ sâu tối đa ~125, an toàn.)

In [18]:
class Solution:
    def basicCalculatorIV(self, expression: str, evalvars: List[str], evalints: List[int]) -> List[str]:
        import re
        env = dict(zip(evalvars, evalints))
        toks = re.findall(r'\d+|[a-z]+|[()+\-*]', expression)
        pos = 0

        def add(p, q, sign):                      # p + sign*q
            r = dict(p)
            for m, c in q.items():
                r[m] = r.get(m, 0) + sign * c
            return {m: c for m, c in r.items() if c}

        def mul(p, q):                            # p * q
            r = {}
            for m1, c1 in p.items():
                for m2, c2 in q.items():
                    m = tuple(sorted(m1 + m2))    # đơn thức chuẩn hoá
                    r[m] = r.get(m, 0) + c1 * c2
            return {m: c for m, c in r.items() if c}

        def atom(t):
            if t.isdigit():
                v = int(t)
            elif t in env:
                v = env[t]
            else:
                return {(t,): 1}
            return {(): v} if v else {}

        def expr():                               # term (('+'|'-') term)*
            nonlocal pos
            v = term()
            while pos < len(toks) and toks[pos] in '+-':
                op = toks[pos]; pos += 1
                v = add(v, term(), 1 if op == '+' else -1)
            return v

        def term():                               # factor ('*' factor)*
            nonlocal pos
            v = factor()
            while pos < len(toks) and toks[pos] == '*':
                pos += 1
                v = mul(v, factor())
            return v

        def factor():                             # số | biến | '(' expr ')'
            nonlocal pos
            t = toks[pos]; pos += 1
            if t == '(':
                v = expr()
                pos += 1                          # bỏ ')'
                return v
            return atom(t)

        poly = expr()
        order = sorted(poly, key=lambda m: (-len(m), m))   # bậc giảm, rồi từ điển
        return ['*'.join([str(poly[m])] + list(m)) for m in order]

for e, v, i, exp in EXAMPLES:
    assert Solution().basicCalculatorIV(e, v, i) == exp, e
print("OK")

OK


Giải thích những dòng quan trọng:

- `re.findall(...)`: token hoá; tên biến dài như `temperature` là một token.
- `{(): v} if v else {}`: hằng số 0 biểu diễn bằng đa thức rỗng — tránh in `"0"`.
- `tuple(sorted(m1 + m2))`: nhân đơn thức = nối danh sách biến rồi sắp.
- Lọc `if c`: mọi kết quả trung gian không chứa hệ số 0.
- `'*'.join([str(c)] + list(m))`: hằng số (`m = ()`) chỉ in hệ số; còn lại in `c*v1*v2...`.

## 10. Vì sao thuật toán đúng

1. **Đúng cú pháp.** Văn phạm `expr/term/factor` là văn phạm chuẩn cho biểu thức với hai mức ưu tiên và kết hợp trái; mỗi hàm tiêu thụ đúng phần chuỗi tương ứng (quy nạp theo độ dài). Shunting-yard với `>=` tạo cùng cây cú pháp.
2. **Đúng ngữ nghĩa.** Mỗi đa thức (dict) biểu diễn **duy nhất** một đa thức theo nghĩa toán học, vì: đơn thức được chuẩn hoá (tuple đã sắp) và không có hệ số 0. `add` và `mul` cài đúng định nghĩa phép cộng và phép nhân đa thức (luật phân phối + giao hoán của biến).
3. **Thay biến.** Thay `x` bằng hằng số `k` trước khi tính tương đương thay sau khi khai triển (phép đồng cấu đánh giá giao hoán với `+`, `*`).
4. **Định dạng.** Khoá sắp `(-len(m), m)` đúng là "bậc giảm dần, rồi từ điển trên danh sách biến"; hằng số có bậc 0 đứng cuối. Dạng chuẩn duy nhất ⇒ output duy nhất.

## 11. Phân tích Time/Space

Gọi `T` = số token (≤ 125), `M` = số đơn thức tối đa trong một đa thức trung gian, `D` = bậc tối đa.

| Cách | Time | Space |
|---|---|---|
| 1. Chưa gom | mũ theo số phép nhân lồng nhau | mũ |
| 2. Đệ quy + dict | `O(T · M^2 · D log D)` | `O(M · D)` + đệ quy `O(T)` |
| 3. Hai ngăn xếp | `O(T · M^2 · D log D)` | `O(T · M · D)` |
| 4. Nạp chồng + eval | như 2 | như 2 |

In [19]:
heavy = " * ".join(["(a + b + c + 1)"] * 6)
for name, f in [("brute", brute), ("rd", calc_rd), ("stack", calc_stack), ("eval", calc_eval)]:
    t0 = time.perf_counter()
    r = f(heavy, [], [])
    print(f"{name:6} {len(r):4} hạng tử  {1000 * (time.perf_counter() - t0):8.1f} ms")

brute    84 hạng tử       4.5 ms
rd       84 hạng tử       0.5 ms
stack    84 hạng tử       0.5 ms
eval     84 hạng tử       0.7 ms


## 12. Bộ test

Ngoài các ví dụ, ta có một bộ kiểm tra **độc lập** rất mạnh: với đa thức kết quả, **thế các giá trị ngẫu nhiên** cho biến và so với giá trị Python tính trực tiếp từ biểu thức gốc (thay biến bằng số rồi `eval` số học). Nếu khai triển sai, gần như chắc chắn hai giá trị lệch nhau.

In [20]:
methods = {
    "brute": brute, "rd": calc_rd, "stack": calc_stack, "eval": calc_eval,
    "SolutionRD": lambda e, v, i: SolutionRD().basicCalculatorIV(e, v, i),
    "SolutionStack": lambda e, v, i: SolutionStack().basicCalculatorIV(e, v, i),
    "SolutionEval": lambda e, v, i: SolutionEval().basicCalculatorIV(e, v, i),
    "Solution": lambda e, v, i: Solution().basicCalculatorIV(e, v, i),
}

# 12.1 Ví dụ
for name, f in methods.items():
    for e, v, i, exp in EXAMPLES:
        assert f(e, v, i) == exp, (name, e)

# 12.2 Edge cases
edge = [
    ("0", [], [], []),
    ("5", [], [], ["5"]),
    ("x", [], [], ["1*x"]),
    ("x", ["x"], [0], []),
    ("x - x", [], [], []),
    ("a * 0 + b", [], [], ["1*b"]),
    ("b * a - a * b", [], [], []),
    ("aa * a + a * aa", [], [], ["2*a*aa"]),          # 'a' < 'aa' theo từ điển
    ("(((x)))", [], [], ["1*x"]),
    ("x * x * y - y * x * x + z", [], [], ["1*z"]),
    ("e * e", ["e"], [-3], ["9"]),
]
for name, f in methods.items():
    for e, v, i, exp in edge:
        assert f(e, v, i) == exp, (name, e, f(e, v, i))
print("Ví dụ + edge cases: OK")

Ví dụ + edge cases: OK


In [21]:
# 12.3 Biểu thức ngẫu nhiên: so sánh các cách + kiểm tra bằng thế số
import random
random.seed(770)
VARS = ["a", "b", "c", "x", "yy"]

def rand_expr(depth=0):
    r = random.random()
    if depth >= 3 or r < 0.3:
        return str(random.randint(0, 9)) if random.random() < 0.4 else random.choice(VARS)
    if r < 0.45:
        return "(" + rand_expr(depth + 1) + ")"
    op = random.choice(["+", "-", "*"])
    return rand_expr(depth + 1) + f" {op} " + rand_expr(depth + 1)

def eval_poly_terms(terms, values):
    total = 0
    for term in terms:
        parts = term.split('*')
        v = int(parts[0])
        for name in parts[1:]:
            v *= values[name]
        total += v
    return total

def eval_numeric(expression, values):
    toks = TOKEN_RE.findall(expression)
    code = ' '.join(str(values[t]) if t.isalpha() else t for t in toks)
    return eval(code, {"__builtins__": {}}, {})

for _ in range(400):
    e = rand_expr()
    k = random.randint(0, 2)
    ev = random.sample(VARS, k)
    ei = [random.randint(-5, 5) for _ in ev]
    outs = {name: tuple(f(e, ev, ei)) for name, f in methods.items()}
    assert len(set(outs.values())) == 1, (e, outs)
    res = list(outs["Solution"])
    for _ in range(3):
        vals = {x: random.randint(-7, 7) for x in VARS}
        vals.update(dict(zip(ev, ei)))
        assert eval_poly_terms(res, vals) == eval_numeric(e, vals), (e, res)
print("400 biểu thức ngẫu nhiên: các cách khớp nhau và khớp thế số")

400 biểu thức ngẫu nhiên: các cách khớp nhau và khớp thế số


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Không sắp biến trong đơn thức → `a*b` và `b*a` không gộp

**Code sai:**

In [22]:
def wrong_mul_nosort(p, q):
    r = {}
    for m1, c1 in p.items():
        for m2, c2 in q.items():
            m = m1 + m2                       # SAI: không sắp
            r[m] = r.get(m, 0) + c1 * c2
    return r

ab = wrong_mul_nosort({('a',): 1}, {('b',): 1})
ba = wrong_mul_nosort({('b',): 1}, {('a',): 1})
print("sai:", p_add(ab, ba, -1), "  đúng:", p_add(p_mul({('a',): 1}, {('b',): 1}), p_mul({('b',): 1}, {('a',): 1}), -1))

sai: {('a', 'b'): 1, ('b', 'a'): -1}   đúng: {}


**Vì sao sai:** `('a','b')` và `('b','a')` là hai khoá khác nhau → `a*b - b*a` không triệt tiêu.

**Code đúng:** `tuple(sorted(m1 + m2))`.

### Lỗi 2 — Sắp kết quả bằng chuỗi đã định dạng hoặc theo hệ số

**Code sai:**

In [23]:
def wrong_format(p):
    terms = ['*'.join([str(c)] + list(m)) for m, c in p.items() if c]
    return sorted(terms, key=lambda s: (-s.count('*'), s))     # SAI: so sánh cả hệ số

p = {('a', 'b'): 3, ('a', 'c'): -5, ('b', 'b'): 1, (): 2}
print("sai: ", wrong_format(p))
print("đúng:", p_format(p))

sai:  ['-5*a*c', '1*b*b', '3*a*b', '2']
đúng: ['3*a*b', '-5*a*c', '1*b*b', '2']


**Vì sao sai:** so sánh chuỗi `"-5*a*c"` với `"3*a*b"` bị ảnh hưởng bởi **hệ số** (`'-' < '3'`), trong khi đề yêu cầu so sánh **danh sách biến**.

**Code đúng:** sắp theo khoá `(-len(m), m)` trên **đơn thức**, rồi mới định dạng.

### Lỗi 3 — Quên bỏ hạng tử hệ số 0

**Code sai:**

In [24]:
def wrong_keep_zero(expression):
    toks = TOKEN_RE.findall(expression)
    pos = 0
    def add(p, q, s):
        r = dict(p)
        for m, c in q.items():
            r[m] = r.get(m, 0) + s * c        # SAI: không xoá hệ số 0
        return r
    def expr():
        nonlocal pos
        v = {(toks[pos],) if toks[pos].isalpha() else (): 1 if toks[pos].isalpha() else int(toks[pos])}
        pos += 1
        while pos < len(toks):
            op = toks[pos]; t = toks[pos + 1]; pos += 2
            v = add(v, {(t,) if t.isalpha() else (): 1 if t.isalpha() else int(t)}, 1 if op == '+' else -1)
        return v
    p = expr()
    return ['*'.join([str(p[m])] + list(m)) for m in sorted(p, key=lambda m: (-len(m), m))]

print("sai:", wrong_keep_zero("a - a + 7 - 7"), " đúng:", calc_rd("a - a + 7 - 7", [], []))

sai: ['0*a', '0']  đúng: []


**Vì sao sai:** đề yêu cầu **bỏ** hạng tử có hệ số 0; `"7 - 7"` phải trả `[]`, không phải `["0"]`.

**Code đúng:** lọc `{m: c for m, c in r.items() if c}` sau mỗi phép toán, và hằng số 0 → `{}`.

### Lỗi 4 — Xử lý `-` như `+` rồi đổi dấu chỉ một nhân tử / sai kết hợp

**Code sai:**

In [25]:
def wrong_right_assoc(expression):
    """Shunting-yard nhưng dùng '>' thay vì '>=' -> kết hợp phải."""
    vals, ops = [], []
    for t in TOKEN_RE.findall(expression):
        if t == '(':
            ops.append(t)
        elif t == ')':
            while ops[-1] != '(':
                apply_op(vals, ops.pop())
            ops.pop()
        elif t in PREC:
            while ops and ops[-1] != '(' and PREC[ops[-1]] > PREC[t]:    # SAI
                apply_op(vals, ops.pop())
            ops.append(t)
        else:
            vals.append(make_atom(t, {}))
    while ops:
        apply_op(vals, ops.pop())
    return p_format(vals[-1])

print("sai:", wrong_right_assoc("a - b - c"), " đúng:", calc_stack("a - b - c", [], []))

sai: ['1*a', '-1*b', '1*c']  đúng: ['1*a', '-1*b', '-1*c']


**Vì sao sai:** với `>`, `a - b - c` bị tính thành `a - (b - c) = a - b + c`.

**Code đúng:** `PREC[ops[-1]] >= PREC[t]` (kết hợp trái).

## 14. Mẹo & nhận dạng pattern

- **Biểu thức có độ ưu tiên + ngoặc** → văn phạm `expr/term/factor` hoặc shunting-yard (*224*, *227*, *772*).
- **"Giá trị" không phải số** (đa thức, ma trận, tập hợp…) → giữ nguyên bộ phân tích, chỉ thay các phép toán.
- **Đa thức nhiều biến** → `dict` {tuple biến đã sắp: hệ số}; luôn chuẩn hoá và xoá hệ số 0.
- **Thứ tự in phức tạp** → sắp trên **cấu trúc dữ liệu**, định dạng sau cùng.
- **Kiểm thử khai triển đa thức** → thế số ngẫu nhiên (Schwartz–Zippel: hai đa thức khác nhau hiếm khi trùng giá trị tại điểm ngẫu nhiên).

## 15. Biến thể

1. **Basic Calculator I/II/III** (*224, 227, 772*): giá trị là số nguyên.
2. **Có lũy thừa `^`** (ưu tiên cao hơn `*`, kết hợp phải) → thêm một tầng văn phạm `power := factor ('^' power)?`.
3. **Có dấu trừ một ngôi** → `factor := '-' factor | ...`.
4. **Đạo hàm theo một biến** → duyệt đa thức kết quả: mỗi đơn thức có `k` lần biến `x` → hệ số nhân `k`, bớt một `x`.
5. **Đánh giá đa thức tại nhiều điểm** → sau khi rút gọn, thế số.

In [26]:
# Biến thể 4: đạo hàm theo x
def derivative(poly, x):
    r = {}
    for m, c in poly.items():
        k = m.count(x)
        if k:
            lst = list(m); lst.remove(x)
            key = tuple(lst)
            r[key] = r.get(key, 0) + c * k
    return {m: c for m, c in r.items() if c}

def parse_poly(expression, evalvars=(), evalints=()):
    env = dict(zip(evalvars, evalints))
    vals, ops = [], []
    for t in TOKEN_RE.findall(expression):
        if t == '(':
            ops.append(t)
        elif t == ')':
            while ops[-1] != '(':
                apply_op(vals, ops.pop())
            ops.pop()
        elif t in PREC:
            while ops and ops[-1] != '(' and PREC[ops[-1]] >= PREC[t]:
                apply_op(vals, ops.pop())
            ops.append(t)
        else:
            vals.append(make_atom(t, env))
    while ops:
        apply_op(vals, ops.pop())
    return vals[-1]

poly = parse_poly("(x + 1) * (x + 1) * y + 3 * x")
print("f      =", p_format(poly))
print("df/dx  =", p_format(derivative(poly, "x")))

f      = ['1*x*x*y', '2*x*y', '3*x', '1*y']
df/dx  = ['2*x*y', '2*y', '3']


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chưa gom | Danh sách hạng tử, gom cuối | Mũ | Mũ | Kiểm thử biểu thức nhỏ |
| 2. Đệ quy đi xuống | `expr/term/factor` trên dict | `O(T·M²·D log D)` | `O(M·D)` | Nộp bài, dễ đọc |
| 3. Hai ngăn xếp | Shunting-yard trên dict | như 2 | `O(T·M·D)` | Tránh đệ quy |
| 4. Nạp chồng + eval | Lớp `Poly` + `eval` | như 2 | như 2 | Viết nhanh, minh hoạ |

## 17. Bài tập

**Bài 1 (đoán output).** `basicCalculatorIV("a * b * c + b * a * c * 4", [], [])` và `basicCalculatorIV("(a + b) * (a - b)", ["b"], [2])` = ?

**Bài 2 (tính tay).** Khai triển và in đúng thứ tự: `"(x + y) * (x + y) - x * x"`.

**Bài 3 (điền TODO).** Hoàn thiện `p_sub(p, q)` không dùng `p_add`.

**Bài 4 (tự viết lại).** Viết lại `factor()` để hỗ trợ **dấu trừ một ngôi**: `"- a * - b"` → `["1*a*b"]`.

**Bài 5 (mini challenge).** Thêm toán tử lũy thừa `^` với số mũ là số nguyên không âm (kết hợp phải, ưu tiên cao hơn `*`).

In [27]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def p_sub(p, q):
    # TODO: r = dict(p); với mỗi (m, c) trong q: r[m] -= c; xoá nếu 0
    pass

# Bài 4 — TODO
def calc_unary(expression):
    # TODO: unary := '-' unary | factor;  factor := NUMBER | VAR | '(' expr ')'
    pass

# Bài 5 — TODO
def calc_power(expression):
    # TODO: term := power ('*' power)*; power := factor ('^' power)?
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `b*a*c` sau khi sắp là `a*b*c`.
- Mức 2: `1 + 4 = 5`.
- Mức 3: `["5*a*b*c"]`; `(a + 2)(a - 2) = a² − 4` → `["1*a*a", "-4"]`.

**Bài 2.**
- Mức 1: `(x+y)² = x² + 2xy + y²`.
- Mức 2: trừ `x²` → `2xy + y²`.
- Mức 3: cùng bậc 2: `('x','y') < ('y','y')` → `["2*x*y", "1*y*y"]`.

**Bài 3.**
- Mức 1: sao chép `p`.
- Mức 2: `r[m] = r.get(m, 0) - c`.
- Mức 3: xoá khoá nếu về 0.

**Bài 4.**
- Mức 1: ở tầng ngay trên `factor` (tầng `power` nếu có `^`), nếu token là `'-'` thì đọc tiếp tầng đó một lần nữa.
- Mức 2: nhân kết quả với `{(): -1}`.
- Mức 3: `"- a * - b"` = `(-a) * (-b)`.

**Bài 5.**
- Mức 1: thêm tầng `power` giữa `term` và `factor`.
- Mức 2: `power := factor ('^' power)?` (đệ quy bên phải → kết hợp phải).
- Mức 3: lũy thừa = nhân lặp; số mũ đọc từ đa thức hằng `{(): k}`.

## 19. Đáp án tham khảo

In [28]:
# Bài 1
assert Solution().basicCalculatorIV("a * b * c + b * a * c * 4", [], []) == ["5*a*b*c"]
assert Solution().basicCalculatorIV("(a + b) * (a - b)", ["b"], [2]) == ["1*a*a", "-4"]

# Bài 2
assert Solution().basicCalculatorIV("(x + y) * (x + y) - x * x", [], []) == ["2*x*y", "1*y*y"]

# Bài 3
def p_sub_ans(p, q):
    r = dict(p)
    for m, c in q.items():
        r[m] = r.get(m, 0) - c
        if r[m] == 0:
            del r[m]
    return r
assert p_sub_ans({('a',): 2, (): 1}, {('a',): 2}) == {(): 1}

# Bài 4 + Bài 5: một bộ phân tích có dấu trừ một ngôi và lũy thừa
TOK2 = re.compile(r'\d+|[a-z]+|[()+\-*^]')

def calc_ext(expression):
    toks = TOK2.findall(expression)
    pos = 0
    def peek():
        return toks[pos] if pos < len(toks) else None
    def expr():
        nonlocal pos
        v = term()
        while peek() in ('+', '-'):
            op = toks[pos]; pos += 1
            v = p_add(v, term(), 1 if op == '+' else -1)
        return v
    def term():
        nonlocal pos
        v = power()
        while peek() == '*':
            pos += 1
            v = p_mul(v, power())
        return v
    def power():
        nonlocal pos
        if peek() == '-':                     # trừ một ngôi: -x^2 = -(x^2)
            pos += 1
            return p_mul({(): -1}, power())
        base = factor()
        if peek() == '^':
            pos += 1
            ex = power()                      # kết hợp phải
            k = ex.get((), 0)
            r = {(): 1}
            for _ in range(k):
                r = p_mul(r, base)
            return r
        return base
    def factor():
        nonlocal pos
        t = toks[pos]; pos += 1
        if t == '(':
            v = expr(); pos += 1
            return v
        return make_atom(t, {})
    return p_format(expr())

assert calc_ext("- a * - b") == ["1*a*b"]
assert calc_ext("(x + 1) ^ 2") == ["1*x*x", "2*x", "1"]
assert calc_ext("2 ^ 3 ^ 2") == ["512"]                 # 2^(3^2)
assert calc_ext("- (a - b) ^ 2 + a * a") == ["2*a*b", "-1*b*b"]
random.seed(5)
for _ in range(200):
    e = rand_expr()
    assert calc_ext(e) == calc_rd(e, [], [])           # không có '^', '-' một ngôi -> như cũ
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
toks = re.findall(r'\d+|[a-z]+|[()+\-*]', expression)
# đa thức: {tuple_biến_đã_sắp: hệ_số}, không chứa hệ số 0
def expr():  v = term();   while tok in '+-': v = add(v, term(), ±1)
def term():  v = factor(); while tok == '*':  v = mul(v, factor())
def factor(): '(' expr ')' | số -> {(): n} | biến -> {(): env[x]} hoặc {(x,): 1}
mul: key = tuple(sorted(m1 + m2))
output: sorted(poly, key=lambda m: (-len(m), m)) -> '*'.join([str(c)] + list(m))
```

**Checklist trước khi submit:**

- [ ] Đơn thức là **tuple đã sắp**.
- [ ] Xoá hệ số 0 sau **mọi** phép toán; hằng số 0 → `{}`.
- [ ] Thay biến `evalvars` ngay khi đọc token.
- [ ] Thứ tự: `(-bậc, danh sách biến)`; hằng số cuối.
- [ ] In hệ số kể cả `1`/`-1`: `"1*e*e"`.
- [ ] `-` kết hợp trái (`>=` trong shunting-yard).
- [ ] Tên method đúng: `basicCalculatorIV(self, expression, evalvars, evalints)`.